🛰️ Green Area Analyzer using Google Earth Engine & Machine Learning
==================================================================

📌 Project Overview:
This notebook is designed to monitor and forecast vegetation trends by analyzing NDVI (Normalized Difference Vegetation Index) over time using satellite imagery from the Sentinel-2 dataset, accessed via the Google Earth Engine API.

The tool processes satellite images for a user-defined region and time period, extracts NDVI values, and uses various regression models to predict the greenness of that area in future years. It supports trend visualization and comparative analysis using multiple ML models.

🌿 Use Case:
This project is useful for environmental monitoring, deforestation tracking, agricultural assessment, and climate change studies. It provides a simple pipeline to understand how green cover is changing and how it might evolve in the coming years.

--------------------------------------------------------
🧠 Machine Learning Models Used:
--------------------------------------------------------
1. Random Forest Regressor (Robust and interpretable)
2. Support Vector Regressor (Works well with nonlinear data)
3. Gradient Boosting Regressor (Accurate, slow but powerful)
4. Decision Tree Regressor (Simple and fast)
5. XGBoost Regressor (Optimized and scalable boosting)

--------------------------------------------------------
📦 Python Libraries Required:
--------------------------------------------------------
- earthengine-api: Interface to Google Earth Engine
- numpy: Array manipulation
- pandas: Dataframe operations
- matplotlib: Data visualization
- scikit-learn: Machine learning algorithms
- xgboost: Gradient boosting model
- mpl_toolkits.axes_grid1: Advanced plotting (included with matplotlib)

--------------------------------------------------------
🛠️ Workflow:
--------------------------------------------------------
1. Authenticate and initialize the Earth Engine API.
2. Select location coordinates (latitude-longitude bounding box).
3. Fetch Sentinel-2 imagery filtered by date and cloud coverage.
4. Compute NDVI from bands B8 (NIR) and B4 (Red).
5. Store NDVI data with timestamps.
6. Train multiple regression models on historical NDVI data.
7. Predict NDVI values for future time frames.
8. Plot original data, fitted model, and forecasted trend.
9. Automatically classify the vegetation trend (increasing/decreasing).

--------------------------------------------------------
📅 Parameters:
--------------------------------------------------------
- `locations`: Dictionary with region names and bounding box coordinates.
- `train_years`: Number of years to train the model on historical data.
- `predict_years`: Number of years to forecast into the future.

--------------------------------------------------------
📍 Example Use:
--------------------------------------------------------
- Analyzing forest greenness over the Amazon basin from 2019–2025.
- Predicting NDVI trends to monitor possible deforestation or seasonal variation.
- Comparing the performance of different ML models for environmental forecasting.

--------------------------------------------------------
👨‍💻 Author: Chinmay Bansal
📅 Year: 2025

In [ ]:
%pip install earthengine-api numpy pandas matplotlib scikit-learn xgboost
%pip install --upgrade xgboost

In [ ]:
import ee
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler

In [ ]:
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='satellite-454512')

In [ ]:
class GreenAreaAnalyzer:
    def __init__(self, locations, train_years=5, predict_years=2):
        self.locations = locations
        self.train_years = train_years
        self.predict_years = predict_years

    def fetch_and_calculate_ndvi(self, name, coords, start_date, end_date):
        area_of_interest = ee.Geometry.Rectangle([coords[1], coords[0], coords[3], coords[2]])
        collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                      .filterBounds(area_of_interest)
                      .filterDate(start_date, end_date)
                      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
                      .select(['B8', 'B4']))
        images = collection.toList(collection.size()).getInfo()

        if len(images) == 0:
            print(f"No images found for {name} in the given date range.")
            return name, pd.DataFrame()

        ndvi_values, dates = [], []
        for image_info in images:
            image = ee.Image(image_info['id'])
            timestamp = image_info['properties']['system:time_start'] / 1000
            date = datetime.fromtimestamp(timestamp)
            ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
            result = ndvi.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=area_of_interest,
                scale=30,
                maxPixels=1e13
            ).getInfo()
            ndvi_value = result.get('NDVI')
            if ndvi_value is not None:
                ndvi_values.append(float(ndvi_value))
                dates.append(date)

        return name, pd.DataFrame({'Date': dates, 'NDVI': ndvi_values})

    def analyze_sequential(self, start_date, end_date):
        results = {}
        for loc in self.locations.items():
            name, data = self.fetch_and_calculate_ndvi(loc[0], loc[1], start_date, end_date)
            results[name] = data
        return results

    def plot_and_predict(self):
        base_year = 2019
        train_start_date = datetime(base_year, 1, 1).strftime('%Y-%m-%d')
        train_end_date = datetime(base_year + self.train_years, 1, 1).strftime('%Y-%m-%d')
        predict_end_date = datetime(base_year + self.train_years + self.predict_years, 1, 1).strftime('%Y-%m-%d')

        results = self.analyze_sequential(train_start_date, train_end_date)
        actual_future_results = self.analyze_sequential(train_end_date, predict_end_date)

        regressors = {
            'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42),
            'SVR': SVR(),
            'GradientBoosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
            'DecisionTree': DecisionTreeRegressor(random_state=42),
            'XGBoost': XGBRegressor(n_estimators=100, random_state=42)
        }

        for name, df in results.items():
            if df.empty:
                print(f"No data available for {name}.")
                continue

            df['Timestamp'] = pd.to_datetime(df['Date']).astype('int64') / 10**9
            X = df['Timestamp'].values.reshape(-1, 1)
            y = df['NDVI'].values.reshape(-1, 1)

            scaler_X = StandardScaler()
            scaler_y = StandardScaler()
            X_scaled = scaler_X.fit_transform(X)
            y_scaled = scaler_y.fit_transform(y)

            future_dates = pd.date_range(start=train_end_date, end=predict_end_date, freq='ME')
            future_timestamps = (future_dates - pd.Timestamp("1970-01-01")) // pd.Timedelta('1s')
            future_timestamps = np.array(future_timestamps).reshape(-1, 1)
            future_timestamps_scaled = scaler_X.transform(future_timestamps)

            n_rows = len(regressors)
            fig, axs = plt.subplots(n_rows, 2, figsize=(20, 6 * n_rows), squeeze=False)

            train_trend = 'increasing' if y[-1][0] > y[0][0] else 'decreasing'
            print(f"🌿 Greenery Trend for {name} based on Training Data: {train_trend}")

            for i, (reg_name, regressor) in enumerate(regressors.items()):
                regressor.fit(X_scaled, y_scaled.ravel())

                ax = axs[i, 0]
                ax.scatter(df['Date'], y.flatten(), color='blue', label='Actual NDVI', s=30)

                fitted = scaler_y.inverse_transform(
                    regressor.predict(X_scaled).reshape(-1, 1)
                ).flatten()
                ax.plot(df['Date'], fitted, color='orange', label='Fitted NDVI', linewidth=1.5)

                predictions_scaled = regressor.predict(future_timestamps_scaled)
                predictions = scaler_y.inverse_transform(predictions_scaled.reshape(-1, 1)).flatten()

                ax.plot(future_dates, predictions, color='green', label='Predicted NDVI', linewidth=1.5)
                ax.set_title(f'{reg_name} - {name}')
                ax.legend()
                ax.grid(True)

                trend = 'increasing' if predictions[-1] > predictions[0] else 'decreasing'

                ax2 = axs[i, 1]
                ax2.plot(future_dates, predictions,
                         color='green' if trend == 'increasing' else 'red', marker='o')
                ax2.set_title(f'Greenery Trend - {reg_name} ({trend})')
                ax2.grid(True)
                ax2.set_yticks(np.arange(min(predictions) - 0.005, max(predictions) + 0.005, 0.005))

            plt.tight_layout()
            plt.show()

In [ ]:
if __name__ == "__main__":
  locations = {
    'Amazon Rainforest (Brazil)': [-3.5, -60.0, -3.0, -59.5],
    'Ganges Delta (India/Bangladesh)': [22.0, 89.0, 22.5, 89.5],
  }

  analyzer = GreenAreaAnalyzer(locations)
  analyzer.plot_and_predict()

## Conclusion and evaluation note

The original plots compare fitted curves and future projections, but visual inspection alone does not establish which model forecasts best. Use the **Chronological Holdout Evaluation** section below to compare Prophet and the regression models against later, observed satellite NDVI values using MAE, RMSE, and R². The best model should be selected from the holdout results, not assumed in advance.


# Chronological Holdout Evaluation: Prophet vs. Regression Models

This section evaluates forecasts on observations that are **not used for fitting**. It uses the same Sentinel-2 NDVI extraction method defined above and aggregates observations to monthly mean NDVI so all models are compared at the same monthly frequency.

**Default periods**
- Training: 2019-01-01 through 2023-12-31 (five years)
- Test: 2024-01-01 through 2025-12-31 (two years)
- Cloud filter: Sentinel-2 `CLOUDY_PIXEL_PERCENTAGE < 10` (from the existing extraction method)
- Spatial reduction: mean NDVI over the selected region, scale 30 m (from the existing extraction method)

The test interval is kept separate from training. Metrics are calculated only for test months that contain observed NDVI data. Run this section after the class definition above. Earth Engine authentication and access to the selected region are required.

In [ ]:
%pip install prophet

## Run the holdout evaluation

The code below trains Prophet, Random Forest, SVR, Gradient Boosting, Decision Tree, and XGBoost on the training period, predicts NDVI for the observed test-month dates, and prints the model parameters and holdout metrics. The feature scaler is fitted on training timestamps only. No test observations are used to fit the models.

In [ ]:
from prophet import Prophet
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import warnings

warnings.filterwarnings("default")

# Reproducible chronological split; adjust only if you have a justified reason.
EVAL_START = "2019-01-01"
TRAIN_END = "2024-01-01"
TEST_END = "2026-01-01"
MONTH_FREQ = "ME"

# Reuse the existing Sentinel-2 / NDVI extraction implementation.
eval_analyzer = GreenAreaAnalyzer(locations, train_years=5, predict_years=2)

train_raw = eval_analyzer.analyze_sequential(EVAL_START, TRAIN_END)
test_raw = eval_analyzer.analyze_sequential(TRAIN_END, TEST_END)

def monthly_ndvi(raw_df):
    if raw_df is None or raw_df.empty:
        return pd.DataFrame(columns=["ds", "y"])
    data = raw_df[["Date", "NDVI"]].copy()
    data["Date"] = pd.to_datetime(data["Date"], errors="coerce")
    data["NDVI"] = pd.to_numeric(data["NDVI"], errors="coerce")
    data = data.dropna().sort_values("Date").set_index("Date")
    monthly = data["NDVI"].resample(MONTH_FREQ).mean().dropna()
    return monthly.rename_axis("ds").reset_index(name="y")

model_specs = {
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "SVR": make_pipeline(StandardScaler(), SVR(kernel="rbf", C=1.0, epsilon=0.1, gamma="scale")),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42, objective="reg:squarederror", eval_metric="rmse")
}

all_metrics = []
all_predictions = {}
all_parameters = {}

for location_name in locations:
    print("\\n" + "=" * 80)
    print(f"LOCATION: {location_name}")
    train_monthly = monthly_ndvi(train_raw.get(location_name))
    test_monthly = monthly_ndvi(test_raw.get(location_name))

    train_monthly = train_monthly[
        (train_monthly["ds"] >= pd.Timestamp(EVAL_START)) &
        (train_monthly["ds"] < pd.Timestamp(TRAIN_END))
    ].copy()
    test_monthly = test_monthly[
        (test_monthly["ds"] >= pd.Timestamp(TRAIN_END)) &
        (test_monthly["ds"] < pd.Timestamp(TEST_END))
    ].copy()

    if len(train_monthly) < 12 or len(test_monthly) < 3:
        print(f"Skipping: insufficient monthly observations (train={len(train_monthly)}, test={len(test_monthly)}).")
        continue

    print(f"Training monthly observations: {len(train_monthly)}")
    print(f"Test monthly observations: {len(test_monthly)}")
    print(f"Training dates: {train_monthly['ds'].min().date()} to {train_monthly['ds'].max().date()}")
    print(f"Test dates: {test_monthly['ds'].min().date()} to {test_monthly['ds'].max().date()}")

    # Prophet: monthly dates are the ds values and monthly mean NDVI is y.
    prophet_params = {
        "growth": "linear",
        "changepoint_prior_scale": 0.05,
        "yearly_seasonality": True,
        "weekly_seasonality": False,
        "daily_seasonality": False,
        "interval_width": 0.80
    }
    prophet_model = Prophet(**prophet_params)
    prophet_model.fit(train_monthly[["ds", "y"]])
    prophet_forecast = prophet_model.predict(test_monthly[["ds"]])
    prophet_pred = prophet_forecast["yhat"].to_numpy()

    y_test = test_monthly["y"].to_numpy()
    dates_test = test_monthly["ds"].to_numpy()
    location_predictions = pd.DataFrame({
        "ds": test_monthly["ds"].to_numpy(),
        "actual_NDVI": y_test,
        "Prophet_prediction": prophet_pred,
        "Prophet_lower_80": prophet_forecast["yhat_lower"].to_numpy(),
        "Prophet_upper_80": prophet_forecast["yhat_upper"].to_numpy()
    })

    all_parameters[f"{location_name} | Prophet"] = prophet_params
    prophet_mae = mean_absolute_error(y_test, prophet_pred)
    prophet_rmse = np.sqrt(mean_squared_error(y_test, prophet_pred))
    prophet_r2 = r2_score(y_test, prophet_pred) if len(y_test) >= 2 else np.nan
    coverage = np.mean((y_test >= prophet_forecast["yhat_lower"].to_numpy()) &
                       (y_test <= prophet_forecast["yhat_upper"].to_numpy()))
    all_metrics.append({
        "Location": location_name, "Model": "Prophet",
        "Train observations": len(train_monthly), "Test observations": len(test_monthly),
        "MAE": prophet_mae, "RMSE": prophet_rmse, "R2": prophet_r2,
        "80% interval coverage": coverage
    })

    # Regression models use time as the sole predictor; scaler is trained on train dates only.
    train_x = (pd.to_datetime(train_monthly["ds"]).astype("int64") // 10**9).to_numpy().reshape(-1, 1)
    test_x = (pd.to_datetime(test_monthly["ds"]).astype("int64") // 10**9).to_numpy().reshape(-1, 1)
    time_scaler = StandardScaler()
    train_x_scaled = time_scaler.fit_transform(train_x)
    test_x_scaled = time_scaler.transform(test_x)

    for model_name, estimator in model_specs.items():
        # SVR pipeline scales its input internally; other regressors receive standardized timestamps.
        fitted_model = clone(estimator)
        if model_name == "SVR":
            fitted_model.fit(train_x, train_monthly["y"].to_numpy())
            prediction = fitted_model.predict(test_x)
        else:
            fitted_model.fit(train_x_scaled, train_monthly["y"].to_numpy())
            prediction = fitted_model.predict(test_x_scaled)

        location_predictions[f"{model_name}_prediction"] = prediction
        all_parameters[f"{location_name} | {model_name}"] = fitted_model.get_params(deep=True)
        all_metrics.append({
            "Location": location_name, "Model": model_name,
            "Train observations": len(train_monthly), "Test observations": len(test_monthly),
            "MAE": mean_absolute_error(y_test, prediction),
            "RMSE": np.sqrt(mean_squared_error(y_test, prediction)),
            "R2": r2_score(y_test, prediction) if len(y_test) >= 2 else np.nan,
            "80% interval coverage": np.nan
        })

    all_predictions[location_name] = location_predictions

metrics_df = pd.DataFrame(all_metrics)
if metrics_df.empty:
    print("No metrics produced. Check Earth Engine access and whether each period contains enough observations.")
else:
    metrics_df = metrics_df.sort_values(["Location", "RMSE"]).reset_index(drop=True)
    display(metrics_df.round(4))
    print("\\nModel parameters used:")
    for model_key, params in all_parameters.items():
        print(f"\\n{model_key}")
        print(json.dumps(params, indent=2, default=str))

    for location_name, prediction_df in all_predictions.items():
        plt.figure(figsize=(14, 5))
        plt.plot(prediction_df["ds"], prediction_df["actual_NDVI"], marker="o", label="Actual NDVI")
        for model_name in ["Prophet", "Random Forest", "SVR", "Gradient Boosting", "Decision Tree", "XGBoost"]:
            col = f"{model_name}_prediction"
            if col in prediction_df:
                plt.plot(prediction_df["ds"], prediction_df[col], marker=".", label=model_name)
        plt.title(f"Holdout Forecast vs Actual Monthly NDVI — {location_name}")
        plt.xlabel("Month")
        plt.ylabel("Monthly mean NDVI")
        plt.legend(ncol=2)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    print("\\nBest model per location by lowest holdout RMSE:")
    display(metrics_df.loc[metrics_df.groupby("Location")["RMSE"].idxmin()].round(4))

    # Export reproducible results for reporting.
    metrics_df.to_csv("ndvi_holdout_model_metrics.csv", index=False)
    for location_name, prediction_df in all_predictions.items():
        safe_name = "".join(ch if ch.isalnum() else "_" for ch in location_name)
        prediction_df.to_csv(f"ndvi_holdout_predictions_{safe_name}.csv", index=False)
    print("Saved: ndvi_holdout_model_metrics.csv and one predictions CSV per location.")

## How to interpret the results

- **MAE**: mean absolute NDVI error; lower is better.
- **RMSE**: penalizes larger errors; lower is better.
- **R²**: measures explanatory performance on the held-out months; it can be negative.
- **80% interval coverage**: for Prophet, the fraction of actual test values inside the model's nominal 80% prediction interval. It is not an accuracy score.

The model with the lowest test RMSE is the best performer for that location and split, but it should not automatically be described as universally best. For a stronger assessment, use rolling-origin time-series cross-validation and retain a final untouched test period.

**Important:** the notebook must be run with valid Google Earth Engine credentials to obtain the real test metrics. No new metric values are hard-coded here; the displayed numbers are calculated from the fetched satellite observations when these cells run.